# Exploratory Data Analysis

Before building any model, we explore the dataset to understand its structure, identify patterns, and spot potential challenges such as missing metadata or skewed distributions.

## Step 1: Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

interactions = pd.read_csv('../data/interactions_train.csv')
items        = pd.read_csv('../data/items.csv')

print(f'Total interactions: {len(interactions)}')
print(f'Total unique users:  {interactions["u"].nunique()}')
print(f'Total unique books:  {items["i"].nunique()}')


## Step 2: User Activity Distribution

How many books does a typical user borrow? A heavily skewed distribution means most users have very few interactions — a classic challenge for collaborative filtering.

In [ ]:
user_activity = interactions.groupby('u').size()

plt.figure(figsize=(10, 5))
sns.histplot(user_activity, bins=50, color='steelblue', kde=False)
plt.title('User Activity: Books Borrowed per User')
plt.xlabel('Number of Borrowed Books')
plt.ylabel('Number of Users')
plt.xlim(0, 50)
plt.tight_layout()
plt.show()

print(f'Median interactions per user: {user_activity.median():.0f}')
print(f'Mean interactions per user:   {user_activity.mean():.1f}')
print(f'Max interactions per user:    {user_activity.max()}')


## Step 3: Item Popularity — Long Tail

Most books are borrowed very rarely, while a small number of titles dominate. This **long-tail distribution** is typical in library and e-commerce data. Pure popularity-based recommendations would miss the majority of the catalogue.

In [ ]:
item_popularity = interactions.groupby('i').size().sort_values(ascending=False).values

plt.figure(figsize=(10, 5))
plt.plot(item_popularity, color='crimson')
plt.fill_between(range(len(item_popularity)), item_popularity, color='crimson', alpha=0.3)
plt.title('Item Popularity (Long Tail Distribution)')
plt.xlabel('Book Index (most → least popular)')
plt.ylabel('Number of Interactions')
plt.tight_layout()
plt.show()

print(f'Books with only 1 interaction: {(item_popularity == 1).sum()} '
      f'({100*(item_popularity==1).mean():.1f}%)')


## Step 4: Missing Metadata

Content-based filtering relies on book metadata (title, author, subjects). Missing values reduce the quality of TF-IDF representations and motivated our data enrichment efforts.

In [ ]:
missing_pct = (items.isnull().sum() / len(items) * 100).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=missing_pct.index, y=missing_pct.values, palette='viridis')
plt.title('Missing Values in Book Metadata (%)')
plt.ylabel('% Missing')
plt.tight_layout()
plt.show()

print(missing_pct.to_string())


## Key Findings

- **Sparse users**: most users borrowed fewer than 10 books — making it hard for CF to find reliable neighbours.
- **Long tail**: the vast majority of books have very few interactions — justifying the addition of content-based filtering.
- **Missing metadata**: Author and Subjects fields have significant gaps — motivating data enrichment with external APIs.